### Load Data ###

In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report

In [14]:
train_df = pd.read_csv("../data_labelling/train_labeled.csv")
val_df   = pd.read_csv("../data_labelling/val_labeled.csv")
test_df  = pd.read_csv("../data_labelling/test_labeled.csv")

### MERGE ###

In [17]:
train_df["label"] = train_df["label"].astype(int)
val_df["label"]   = val_df["label"].astype(int)
test_df["label"]  = test_df["label"].astype(int)

In [18]:
print(train_df["label"].unique())
print(train_df["label"].dtype)

print(train_df["label"].isna().sum())

[0 2 1]
int64
0


### TOKENIZER ###

In [19]:
from transformers import AutoTokenizer

MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [20]:
def tokenize_texts(texts, tokenizer, max_len=128):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )


In [21]:
train_encodings = tokenize_texts(train_df["cleaned_text"], tokenizer)
val_encodings   = tokenize_texts(val_df["cleaned_text"], tokenizer)
test_encodings  = tokenize_texts(test_df["cleaned_text"], tokenizer)

In [22]:
print(train_encodings.keys())
print(train_encodings["input_ids"].shape)
print(train_encodings["attention_mask"].shape)

KeysView({'input_ids': tensor([[    2, 20374, 30468,  ...,     0,     0,     0],
        [    2, 29190,  3036,  ...,     0,     0,     0],
        [    2,  4184,  1308,  ...,     0,     0,     0],
        ...,
        [    2,  2234,   216,  ...,     0,     0,     0],
        [    2,  4599,  5629,  ...,     0,     0,     0],
        [    2,   286,  4255,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])})
torch.Size([1100, 128])
torch.Size([1100, 128])


### DATASET CLASS ###

In [24]:
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.values

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [25]:
train_dataset = SentimentDataset(train_encodings, train_df["label"])
val_dataset   = SentimentDataset(val_encodings, val_df["label"])
test_dataset  = SentimentDataset(test_encodings, test_df["label"])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

## BASELINE MODEL ##

In [27]:
import torch
from transformers import BertForSequenceClassification
from torch.optim import AdamW

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=3e-5)

In [41]:
from torch.nn import CrossEntropyLoss

class_weights = torch.tensor([1.0, 1.3, 1.3]).to(device)
loss_fn = CrossEntropyLoss(weight=class_weights)

In [42]:
from tqdm import tqdm

def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0

    for batch in tqdm(dataloader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # ⛔ JANGAN pakai labels di model
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        # ✅ hitung loss manual pakai class weight
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [43]:
def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0

    for batch in tqdm(dataloader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # ⛔ JANGAN pakai labels di model
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        # ✅ hitung loss manual pakai class weight
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
EPOCHS = 4

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, device)
    val_loss   = eval_epoch(model, val_loader, device)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss  : {val_loss:.4f}")
    print("-" * 30)

100%|██████████| 16/16 [00:02<00:00,  5.46it/s]


Epoch 1
Train Loss: 0.0355
Val Loss  : 1.5036
------------------------------


100%|██████████| 16/16 [00:02<00:00,  5.41it/s]


Epoch 2
Train Loss: 0.0331
Val Loss  : 1.5790
------------------------------


100%|██████████| 16/16 [00:02<00:00,  5.43it/s]

Epoch 3
Train Loss: 0.0182
Val Loss  : 1.9233
------------------------------


In [45]:
from sklearn.metrics import classification_report
import numpy as np

def get_predictions(model, dataloader, device):
    model.eval()
    preds = []
    labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            batch_preds = torch.argmax(logits, dim=1)

            preds.extend(batch_preds.cpu().numpy())
            labels.extend(batch["labels"].numpy())

    return np.array(preds), np.array(labels)

y_pred, y_true = get_predictions(model, test_loader, device)

print(classification_report(
    y_true,
    y_pred,
    target_names=["Negatif", "Netral", "Positif"]
))

              precision    recall  f1-score   support

     Negatif       0.71      0.80      0.75       130
      Netral       0.49      0.48      0.49        60
     Positif       0.64      0.47      0.54        60

    accuracy                           0.64       250
   macro avg       0.61      0.58      0.59       250
weighted avg       0.64      0.64      0.64       250



### SAVE MODEL 1 ###

In [ ]:
model.save_pretrained("./model1_manual")
tokenizer.save_pretrained("./model1_manual")

('./model1_manual\\tokenizer_config.json',
 './model1_manual\\special_tokens_map.json',
 './model1_manual\\vocab.txt',
 './model1_manual\\added_tokens.json',
 './model1_manual\\tokenizer.json')